# 09 — Live POS Feed *(for the live customer demo)*

Every other notebook in this repo proves a scenario from history that ends at
`AS_OF_DATE = 2025-12-01`. This one is different: it watches a **live, running
simulation** of point-of-sale transactions and shows the same kind of decision
(a stockout on the Aurora Bomber, the viral-product style from notebook 03)
being detected **as it happens**, not read from a table.

## Before running this notebook

In a separate terminal, from the repo root:

```
python scripts/pos_stream_simulator.py
```

Leave it running. It ticks baseline orders across a curated set of stores
carrying Aurora Bomber (plus a few other styles for background variety) and
waits for a trigger. When you're ready for the "viral spike" moment, run in a
*second* terminal:

```
python scripts/pos_stream_trigger.py
```

The simulator and this notebook talk to each other only through parquet
files in `data/stream/` — **neither one ever opens `data/warehouse/retail.duckdb`
in read-write mode**, so the live feed can't corrupt the verified batch
warehouse the rest of the demo relies on. See the plan doc for why (DuckDB
allows one read-write connection *or* many read-only ones per file, never
both at once, across processes).

In [ ]:
import sys
sys.path.insert(0, "../src")
import time
from pathlib import Path

import duckdb
import pandas as pd
import plotly.graph_objects as go
from IPython.display import clear_output, display

from retail_synth.viz import CATEGORICAL, STATUS, style_fig

STREAM_DIR = Path("../data/stream")
LIVE_STATE_PATH = STREAM_DIR / "live_state.parquet"
EVENTS_GLOB = str(STREAM_DIR / "events" / "*.parquet")

con = duckdb.connect("../data/warehouse/retail.duckdb", read_only=True)

if not LIVE_STATE_PATH.exists():
    print("No live session found yet. Start the simulator in a separate terminal:\n"
          "  python scripts/pos_stream_simulator.py")
else:
    state = pd.read_parquet(LIVE_STATE_PATH)
    print(f"Live stream found -- tick {int(state['tick'].iloc[0])}, "
          f"{state['location_id'].nunique()} stores, triggered={bool(state['triggered'].iloc[0])}.")
    print("Run the next cell to start watching.")

## Live view

Run the cell below, then trigger the spike from a second terminal whenever you're ready. **Interrupt the kernel (the stop button) to end the live view** — it's designed to keep polling until you stop it, not to run to a fixed completion, so a fixed `MAX_POLLS` below is just a safety net for an unattended session.

In [ ]:
POLL_SECONDS = 2
MAX_POLLS = 120          # ~4 minutes of safety-net polling; interrupt the cell to stop sooner
STOCKOUT_WARNING_ON_HAND = 5

for i in range(MAX_POLLS):
    if not LIVE_STATE_PATH.exists():
        print("Waiting for the simulator to start...")
        time.sleep(POLL_SECONDS)
        continue

    state = pd.read_parquet(LIVE_STATE_PATH)
    events = con.execute(f"""
        SELECT tick, SUM(units) AS units, SUM(gross_revenue) AS revenue
        FROM read_parquet('{EVENTS_GLOB}') GROUP BY tick ORDER BY tick
    """).df()
    recent = events.tail(30)

    aurora = state.loc[state["is_aurora"]].copy()
    by_store = aurora.groupby(["location_id", "store_name", "store_tier"], as_index=False).agg(
        on_hand=("on_hand", "sum"), velocity=("trailing_velocity", "sum")
    ).sort_values("on_hand")

    clear_output(wait=True)

    tick_no = int(state["tick"].iloc[0]) if len(state) else 0
    is_triggered = bool(state["triggered"].iloc[0]) if len(state) else False
    status_label = "TRIGGERED" if is_triggered else "baseline"

    fig1 = go.Figure()
    fig1.add_trace(go.Bar(x=recent["tick"], y=recent["units"], marker_color=CATEGORICAL[0], name="Units/tick"))
    style_fig(fig1, f"Live POS ticker -- tick {tick_no} ({status_label})", height=280)
    display(fig1)

    colors = [STATUS["critical"] if v == 0 else (STATUS["warning"] if v < STOCKOUT_WARNING_ON_HAND else CATEGORICAL[2])
              for v in by_store["on_hand"]]
    fig2 = go.Figure()
    fig2.add_trace(go.Bar(x=by_store["on_hand"], y=by_store["store_name"], orientation="h", marker_color=colors))
    style_fig(fig2, "Aurora Bomber on-hand by store (live)", height=380)
    display(fig2)

    stocked_out = by_store.loc[by_store["on_hand"] == 0, "store_name"].tolist()
    low_stock = by_store.loc[(by_store["on_hand"] > 0) & (by_store["on_hand"] < STOCKOUT_WARNING_ON_HAND), "store_name"].tolist()

    if stocked_out:
        shown = ", ".join(stocked_out[:5]) + (f" (+{len(stocked_out) - 5} more)" if len(stocked_out) > 5 else "")
        print(f"[STOCKOUT] Aurora Bomber is out of stock at {len(stocked_out)} store(s): {shown}")
        print("  Recommended: expedite replenishment / reallocate from slower-moving stores (see notebook 03).")
    elif low_stock:
        print(f"[WATCH] {len(low_stock)} store(s) under {STOCKOUT_WARNING_ON_HAND} units on hand.")
    else:
        print("No live decisions yet -- Aurora Bomber inventory is healthy.")

    time.sleep(POLL_SECONDS)

print("\nStopped (safety-net limit reached). Re-run this cell, or interrupt it any time, to keep watching.")

## What just happened

The trigger you ran produced the same shape of event as the batch-generated
`viral_product` scenario in `config/scenario_config.yaml` — a demand
multiplier applied to the Aurora Bomber's flagship stores — except this time
it played out in front of you, in seconds, against inventory this notebook
watched deplete in real time. That's the difference between *reporting* that
a decision system is "continuous" and *demonstrating* it.